# OKRs 2026Q3

In [3]:
# hide-output
# Import libraries and initialise the BigQuery connector

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np
import plotly.graph_objects as go

bqc = BigQueryConnector()

## Period comparison parameters

Shared by every section's period-over-period comparison (last subsection of each analysis) — defined once here so both periods stay in sync across the whole notebook.

In [4]:
import datetime as dt

period1_start, period1_end = dt.date(2025, 7, 1), dt.date(2025, 9, 30)   # Period 1: Jul-Sep 2025 (Q3)
period2_start, period2_end = dt.date(2026, 1, 1), dt.date(2026, 6, 30)  # Period 2: Jan-Jun 2026

print(f"Period 1: {period1_start} -> {period1_end}")
print(f"Period 2: {period2_start} -> {period2_end}")

Period 1: 2025-07-01 -> 2025-09-30
Period 2: 2026-01-01 -> 2026-06-30


## Get data

### Return rates

In [5]:
# Compute symmetric A/B date windows of equal length anchored on 2026-06-01 (FTUE launch date)

import datetime as dt

start_date = dt.date(2025, 1, 1)
end_date = dt.date(2026, 7, 29)

In [6]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
refresh_data = False

In [7]:
# hide-output
# Estimate query cost for player level + game day SQL before executing

query_location = './sql/returnrate.sql'
parameters = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'), 
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 2.27 GB when run.
Estimated query cost: $0.02


In [8]:
# hide-output
# Fetch player level + game day data from BigQuery or load from local pickle cache
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/returnrate.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/returnrate.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/returnrate.pkl')

In [9]:
data.sort_values(by=['dt'], inplace=True)
data

,dt,dau,returnrate_day_01,dau_d1,returnrate_day_03,dau_d3,returnrate_day_07,dau_d7,returnrate_day_14,dau_d14,...,returnrate_within_01,dau_w1,returnrate_within_03,dau_w3,returnrate_within_07,dau_w7,returnrate_within_14,dau_w14,returnrate_within_28,dau_w28
335,2025-01-01,185977,0.836383,155548,0.803470,149427,0.766132,142483,0.725380,134904,...,0.836383,155548,0.917995,170726,0.943095,175394,0.953397,177310,0.960802,178687
560,2025-01-02,191294,0.834077,159554,0.807260,154424,0.766731,146671,0.725464,138777,...,0.834077,159554,0.918095,175626,0.942899,180371,0.953898,182475,0.961300,183891
230,2025-01-03,190754,0.833414,158977,0.806783,153897,0.767664,146435,0.724588,138218,...,0.833414,158977,0.920730,175633,0.944583,180183,0.955131,182195,0.962538,183608
17,2025-01-04,190089,0.844420,160515,0.803992,152830,0.770355,146436,0.723340,137499,...,0.844420,160515,0.921952,175253,0.945578,179744,0.955947,181715,0.963233,183100
51,2025-01-05,193029,0.830896,160387,0.794679,153396,0.774640,149528,0.731393,141180,...,0.830896,160387,0.912661,176170,0.941967,181827,0.953365,184027,0.961348,185568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,2026-07-25,69570,0.830602,57785,0.805563,56043,NaN,0,NaN,0,...,0.830602,57785,0.913555,63556,NaN,64192,NaN,64192,NaN,64192
469,2026-07-26,71251,0.823188,58653,0.794810,56631,NaN,0,NaN,0,...,0.823188,58653,0.901882,64260,NaN,64260,NaN,64260,NaN,64260
330,2026-07-27,71009,0.835936,59359,NaN,0,NaN,0,NaN,0,...,0.835936,59359,NaN,63432,NaN,63432,NaN,63432,NaN,63432
449,2026-07-28,70223,0.844211,59283,NaN,0,NaN,0,NaN,0,...,0.844211,59283,NaN,59283,NaN,59283,NaN,59283,NaN,59283


### Loyalty segment

In [10]:
# hide-output
# Estimate query cost for player level + game day SQL before executing

query_location = './sql/activity.sql'
parameters = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'), 
    'exclude_networks': []
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 6.58 GB when run.
Estimated query cost: $0.04


In [11]:
# hide-output
# Fetch player level + game day data from BigQuery or load from local pickle cache
loyalty_data = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    loyalty_data = bqc.get(query='./sql/activity.sql', is_path=True, query_parameters=parameters)
    loyalty_data.to_pickle('./data/activity.pkl')
else:
    # Load from local cache to avoid repeated query costs
    loyalty_data = pd.read_pickle('./data/activity.pkl')

In [12]:
loyalty_data

,user_id,dt,dt_week,dt_month,install_dt,install_dt_week,install_dt_month,days_since_install,loyalty_segment,usd_net_iap_revenue,usd_net_ad_revenue
0,93752F4F22B5A9CD,2025-10-15,2025-10-12,2025-10-01,2024-12-18,2024-12-15,2024-12-01,301,1. 26-28 (dedicated),NaN,0.123387
1,FB28110522A079A2,2025-03-23,2025-03-23,2025-03-01,2023-09-09,2023-09-03,2023-09-01,561,3. 04-18 (moderate),NaN,0.016943
2,7FB880617AC04268,2025-10-12,2025-10-12,2025-10-01,2024-04-13,2024-04-07,2024-04-01,547,1. 26-28 (dedicated),NaN,0.112009
3,CB7C0252FE5AC78A,2025-04-26,2025-04-20,2025-04-01,2024-10-07,2024-10-06,2024-10-01,201,1. 26-28 (dedicated),NaN,0.139790
4,55C99BE1E3255D0A,2025-05-24,2025-05-18,2025-05-01,2022-11-10,2022-11-06,2022-11-01,926,3. 04-18 (moderate),NaN,0.009639
...,...,...,...,...,...,...,...,...,...,...,...
73885672,9A104359CBCC77B1,2025-12-07,2025-12-07,2025-12-01,2024-05-28,2024-05-26,2024-05-01,558,1. 26-28 (dedicated),NaN,0.053933
73885673,E001C64794784601,2026-02-10,2026-02-08,2026-02-01,2022-03-29,2022-03-27,2022-03-01,1414,1. 26-28 (dedicated),NaN,NaN
73885674,BD5B39F6914601B4,2025-09-16,2025-09-14,2025-09-01,2023-05-06,2023-04-30,2023-05-01,864,3. 04-18 (moderate),12.414838,NaN
73885675,90B05908B794DB7F,2025-02-13,2025-02-09,2025-02-01,2022-10-29,2022-10-23,2022-10-01,838,4. 01-03 (infrequent),NaN,0.556978


# Return rates

In [13]:
returnrate = data.copy()

In [14]:
# Create a line chart showing return rates over time
fig = px.line(
    returnrate,
    x='dt',
    y=['returnrate_day_01', 'returnrate_day_03', 'returnrate_day_07', 'returnrate_day_14', 'returnrate_day_28'],
    title='Return Rates Over Time',
    labels={'dt': 'Date', 'value': 'Return Rate', 'variable': 'Days'},
    markers=False,
    height=600,
    width=1500
)

#fig.update_layout(hovermode='x unified')
fig.show()

In [15]:
# Create a line chart showing return rates over time
fig = px.line(
    returnrate,
    x='dt',
    y=['returnrate_within_01', 'returnrate_within_03', 'returnrate_within_07', 'returnrate_within_14', 'returnrate_within_28'],
    title='Return Rates Over Time',
    labels={'dt': 'Date', 'value': 'Return Rate', 'variable': 'Days'},
    markers=False,
    height=600,
    width=1500
)

fig.show()

## RRW Forecast & Scenario Analysis

Everything below targets a single metric column — change `metric_col` to retarget any other return-rate horizon (e.g. `returnrate_within_07`).

1. Decompose into trend / weekly seasonal / residual, check stationarity
2. Characterize variance and flag outliers robustly (MAD-based, not std/mean)
3. Detect regime shifts with unsupervised change-point detection (PELT)
4. Fit a 3-month forecast with uncertainty bands, backtested
5. Summary chart + table
6. Standalone "what if return rate got a +X% uplift" scenario overlay

## 1. Data prep + trend/seasonality decomposition

In [16]:
# hide-output
from aux_functions import (
    decompose_series, flag_robust_outliers, detect_changepoints, select_training_start,
    fit_rate_forecast, apply_uplift_scenario, compare_periods,
    plot_stl_decomposition, plot_outliers, plot_changepoints, plot_forecast_summary, plot_uplift_scenario,
    plot_period_comparison,
)

### Parameters

In [17]:
# --- All tunable parameters for this section, in one place ---

metric_col = 'returnrate_within_28'    # which RRW horizon to analyze — swap to retarget (e.g. 'returnrate_within_07')

outlier_threshold = 3.5                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct = 0.005                     # Sec 6: relative uplift applied to the forecast curve (e.g. 0.02 = +2%)

In [18]:
# hide-output
rrw = returnrate[['dt', metric_col]].dropna().copy()
rrw['dt'] = pd.to_datetime(rrw['dt'])
rrw = rrw.set_index('dt').asfreq('D')
n_gaps = rrw[metric_col].isna().sum()
rrw[metric_col] = rrw[metric_col].interpolate()  # fill any single-day calendar gaps

rrw_series = rrw[metric_col]
rrw_series.name = metric_col

print(f"{metric_col}: {len(rrw)} days, {rrw.index.min().date()} -> {rrw.index.max().date()} ({n_gaps} interpolated gap-days)")

returnrate_within_28: 547 days, 2025-01-01 -> 2026-07-01 (0 interpolated gap-days)


In [19]:
# hide-output
rrw_decomp, rrw_adf_stat, rrw_adf_p = decompose_series(rrw_series, period=7)
print(f"ADF test on STL residual: stat={rrw_adf_stat:.3f}, p={rrw_adf_p:.4f} -> {'stationary' if rrw_adf_p < 0.05 else 'NOT stationary'}")

ADF test on STL residual: stat=-7.588, p=0.0000 -> stationary


In [20]:
# Chart — STL decomposition: observed+trend / weekly seasonal / residual
plot_stl_decomposition(rrw_decomp, title=f'{metric_col} — STL decomposition (weekly seasonality)').show()

## 2. Variance & outliers

Variance is measured on the STL residual (raw series variance is dominated by trend/seasonality, which isn't "noise"). Outliers are flagged with a MAD-based robust z-score rather than std/mean, so a handful of extreme days can't inflate the very threshold used to catch them.

**In plain terms:**

- **MAD (Median Absolute Deviation)** is a robust way to measure "how spread out" the data is. It's computed in two steps, both using the *median* rather than the *mean*: (1) find the median of the residual, (2) for every point, measure its distance from that median, then take the median of *those* distances. That's the MAD.
- **Why median instead of mean/std?** The usual way to measure spread — standard deviation around the mean — has a circularity problem: a few extreme values pull the mean toward them and inflate the standard deviation, which makes the extreme values look *less* extreme relative to that inflated yardstick. The median and MAD barely move when a handful of points are extreme, because the median only cares about the middle-ranked value, not the size of the outliers. That's what "robust" means here.
- **The modified z-score** turns "distance from the median" into a standardized score, so it reads on roughly the same scale as a familiar z-score (e.g. "3.5 SDs away"): `0.6745 × (value − median) / MAD`. The `0.6745` constant is there so that, for normally-distributed data, this number lines up with an ordinary z-score — it's just a unit conversion, not a tunable knob.
- **The 3.5 threshold** is a standard rule of thumb (Iglewicz & Hoaglin): a modified z-score beyond ±3.5 flags a point as very unlikely to belong to the same distribution as the bulk of the data. It's a convention, not a law — worth adjusting if it flags too much or too little for this series.

In [21]:
# hide-output
rrw_decomp['modified_z'], rrw_decomp['is_outlier'] = flag_robust_outliers(rrw_decomp['resid'], threshold=outlier_threshold)

cov = rrw_decomp['resid'].std() / rrw_decomp['value'].mean()
print(f"Residual std: {rrw_decomp['resid'].std():.5f}  |  MAD: {(rrw_decomp['resid'] - rrw_decomp['resid'].median()).abs().median():.5f}  |  series mean: {rrw_decomp['value'].mean():.4f}")
print(f"Coefficient of variation (resid std / series mean): {cov:.3%}")
print(f"Outliers flagged (|modified z| > {outlier_threshold}): {rrw_decomp['is_outlier'].sum()} / {len(rrw_decomp)} days")
print()
print("Note: flagged days often cluster in time rather than appearing as isolated spikes — those clusters")
print("usually indicate a regime shift (see change-point section below) rather than independent anomalies.")

rrw_decomp[rrw_decomp['is_outlier']][['value', 'resid', 'modified_z']]

Residual std: 0.00256  |  MAD: 0.00064  |  series mean: 0.9569
Coefficient of variation (resid std / series mean): 0.268%
Outliers flagged (|modified z| > 3.5): 56 / 547 days

Note: flagged days often cluster in time rather than appearing as isolated spikes — those clusters
usually indicate a regime shift (see change-point section below) rather than independent anomalies.


,value,resid,modified_z
dt,,,
2025-01-28,0.946626,-0.010707,-11.280052
2025-01-29,0.941654,-0.016712,-17.599082
2025-01-30,0.946720,-0.008651,-9.115538
2025-02-07,0.942233,-0.013064,-13.760057
2025-02-08,0.945987,-0.006743,-7.107635
2025-02-24,0.962129,0.004555,4.780905
2025-02-25,0.963730,0.006607,6.940718
2025-02-28,0.946253,-0.009595,-10.109726
2025-03-01,0.935331,-0.016628,-17.510898


In [22]:
# Chart — observed series with flagged outlier days marked
plot_outliers(rrw_decomp, rrw_decomp['is_outlier'], title=f'{metric_col} with robust-outlier days flagged').show()

## 3. Change-point detection (regime shifts)

Unsupervised structural-break detection (PELT, `ruptures`) on the deseasonalized series — no prior list of "known events" required. Detects level shifts (a permanent-looking step, e.g. a feature launch) rather than single-day spikes, which is what the outlier check above already covers. The series is standardized first since the penalty is scale-sensitive; `changepoint_penalty` defaults to the standard BIC-style `log(n)` — raise it for fewer/more conservative breakpoints, lower it for more sensitivity.

In [23]:
# hide-output
changepoint_dates, changepoint_penalty = detect_changepoints(rrw_decomp['deseasonalized'], min_segment_days=min_segment_days)

print(f"Penalty: {changepoint_penalty:.2f}  |  Change-points detected: {len(changepoint_dates)}")
for d in changepoint_dates:
    print(' ', d.date())

Penalty: 6.30  |  Change-points detected: 10
  2025-01-25
  2025-02-09
  2025-02-24
  2025-05-15
  2025-08-18
  2025-09-02
  2025-12-11
  2025-12-26
  2026-03-26
  2026-06-14


In [24]:
# Chart — observed series with detected change-points marked
plot_changepoints(rrw_decomp, changepoint_dates, title=f'{metric_col} with detected regime changes (dashed lines)').show()

## 4. Forecast (3 months ahead)

**Training window**: the most recent detected regime is often too short to fit weekly seasonality reliably on its own (here, only ~18 days since the last change-point) — so we walk backward through the detected regimes and use the most recent *set* of them that together provide at least `min_training_days`, rather than an arbitrary flat lookback or the single latest (possibly too-short) regime.

**Model**: SARIMAX with weekly seasonality, fit on the **logit-transformed** series rather than the raw rate — this keeps forecast bands naturally within (0, 1). A plain fit on the raw proportion produced a nonsensical >100% upper bound at the 90-day horizon; the logit transform is the standard fix for forecasting bounded rates.

**Backtest**: hold out the last 28 days of the training window, refit, and check forecast error before trusting the live 90-day extrapolation.

In [25]:
# hide-output
# Walk back through detected regimes until we have >= min_training_days (set in Parameters above)
training_start = select_training_start(rrw_decomp.index, changepoint_dates, min_training_days=min_training_days)
train = rrw_series.loc[training_start:]
print(f"Training window: {training_start.date()} -> {rrw_decomp.index.max().date()} ({len(train)} days)")

Training window: 2026-03-26 -> 2026-07-01 (98 days)


In [26]:
# hide-output
backtest_mae, backtest_mape, forecast_df = fit_rate_forecast(
    train, order=sarimax_order, seasonal_order=sarimax_seasonal_order,
    horizon=forecast_horizon_days, backtest_days=backtest_days, ci_alpha=ci_alpha,
)
print(f"Backtest (last {backtest_days} days of training window): MAE={backtest_mae:.5f}  MAPE={backtest_mape:.3%}")

Backtest (last 28 days of training window): MAE=0.00516  MAPE=0.536%


In [27]:
# hide-output
print(f"Forecast horizon: {forecast_df.index.min().date()} -> {forecast_df.index.max().date()}")
for h in [30, 60, 90]:
    row = forecast_df.iloc[h - 1]
    print(f"  +{h}d ({forecast_df.index[h-1].date()}): {row['forecast']:.4f}  [{row['low']:.4f}, {row['high']:.4f}]")

Forecast horizon: 2026-07-02 -> 2026-09-29
  +30d (2026-07-31): 0.9623  [0.9509, 0.9711]
  +60d (2026-08-30): 0.9579  [0.9322, 0.9742]
  +90d (2026-09-29): 0.9609  [0.9195, 0.9815]


## 5. Summary: history + forecast

In [28]:
# Chart — training window history + 90-day forecast with 80% band
plot_forecast_summary(
    rrw_series, train, forecast_df, training_start,
    title=f'{metric_col} — history + {forecast_horizon_days}-day forecast', ci_alpha=ci_alpha,
).show()

In [29]:
# Summary table — forecast at key checkpoints
summary_table = forecast_df.iloc[[29, 59, 89]].reset_index().rename(columns={
    'dt': 'Date', 'forecast': 'P50 (forecast)', 'low': 'P10', 'high': 'P90',
})
summary_table.insert(0, 'Days out', [30, 60, 90])
summary_table[['P10', 'P50 (forecast)', 'P90']] = summary_table[['P10', 'P50 (forecast)', 'P90']].round(4)
summary_table

,Days out,Date,P50 (forecast),P10,P90
0,30,2026-07-31,0.9623,0.9509,0.9711
1,60,2026-08-30,0.9579,0.9322,0.9742
2,90,2026-09-29,0.9609,0.9195,0.9815


## 6. Scenario: what if return rate gets a % uplift

Standalone overlay on the forecast curve itself — applies a relative uplift directly to the forecasted rate (clipped at 100%, since it's a probability), not a downstream DAU/cohort simulation. Adjust `uplift_pct` to try different scenarios.

In [30]:
# hide-output
scenario_df = apply_uplift_scenario(forecast_df, uplift_pct)

print(f"Uplift scenario: {uplift_pct:+.1%} relative")
for h in [30, 60, 90]:
    base = scenario_df['forecast'].iloc[h - 1]
    up = scenario_df['forecast_uplift'].iloc[h - 1]
    print(f"  +{h}d: baseline={base:.4f}  uplifted={up:.4f}  (delta={up - base:+.4f})")

Uplift scenario: +0.5% relative
  +30d: baseline=0.9623  uplifted=0.9671  (delta=+0.0048)
  +60d: baseline=0.9579  uplifted=0.9627  (delta=+0.0048)
  +90d: baseline=0.9609  uplifted=0.9657  (delta=+0.0048)


In [31]:
# Chart — baseline forecast vs uplift scenario
plot_uplift_scenario(
    rrw_series, scenario_df, uplift_pct,
    title=f'{metric_col} — baseline vs {uplift_pct:+.1%} uplift scenario', lookback_days=60,
).show()

## 7. Period comparison: Period 1 vs Period 2

Average `metric_col` within each of the two shared periods defined at the top of the notebook, and the relative uplift between them.

In [32]:
# hide-output
rrw_period_comparison = compare_periods(rrw_series, period1_start, period1_end, period2_start, period2_end)

print(f"Period 1 ({period1_start} -> {period1_end}, n={rrw_period_comparison['period1_n']} days): avg {metric_col} = {rrw_period_comparison['period1_mean']:.4f}")
print(f"Period 2 ({period2_start} -> {period2_end}, n={rrw_period_comparison['period2_n']} days): avg {metric_col} = {rrw_period_comparison['period2_mean']:.4f}")
print(f"Uplift: {rrw_period_comparison['abs_diff']:+.4f}  ({rrw_period_comparison['pct_uplift']:+.1%} relative)")

Period 1 (2025-07-01 -> 2025-09-30, n=92 days): avg returnrate_within_28 = 0.9544
Period 2 (2026-01-01 -> 2026-06-30, n=181 days): avg returnrate_within_28 = 0.9622
Uplift: +0.0078  (+0.8% relative)


In [33]:
# Chart — Period 1 vs Period 2 average
plot_period_comparison(
    rrw_period_comparison, title=f'{metric_col}: Period 1 vs Period 2 average', y_label=metric_col,
).show()

# Loyalty Segments

In [34]:
loyalty_data

,user_id,dt,dt_week,dt_month,install_dt,install_dt_week,install_dt_month,days_since_install,loyalty_segment,usd_net_iap_revenue,usd_net_ad_revenue
0,93752F4F22B5A9CD,2025-10-15,2025-10-12,2025-10-01,2024-12-18,2024-12-15,2024-12-01,301,1. 26-28 (dedicated),NaN,0.123387
1,FB28110522A079A2,2025-03-23,2025-03-23,2025-03-01,2023-09-09,2023-09-03,2023-09-01,561,3. 04-18 (moderate),NaN,0.016943
2,7FB880617AC04268,2025-10-12,2025-10-12,2025-10-01,2024-04-13,2024-04-07,2024-04-01,547,1. 26-28 (dedicated),NaN,0.112009
3,CB7C0252FE5AC78A,2025-04-26,2025-04-20,2025-04-01,2024-10-07,2024-10-06,2024-10-01,201,1. 26-28 (dedicated),NaN,0.139790
4,55C99BE1E3255D0A,2025-05-24,2025-05-18,2025-05-01,2022-11-10,2022-11-06,2022-11-01,926,3. 04-18 (moderate),NaN,0.009639
...,...,...,...,...,...,...,...,...,...,...,...
73885672,9A104359CBCC77B1,2025-12-07,2025-12-07,2025-12-01,2024-05-28,2024-05-26,2024-05-01,558,1. 26-28 (dedicated),NaN,0.053933
73885673,E001C64794784601,2026-02-10,2026-02-08,2026-02-01,2022-03-29,2022-03-27,2022-03-01,1414,1. 26-28 (dedicated),NaN,NaN
73885674,BD5B39F6914601B4,2025-09-16,2025-09-14,2025-09-01,2023-05-06,2023-04-30,2023-05-01,864,3. 04-18 (moderate),12.414838,NaN
73885675,90B05908B794DB7F,2025-02-13,2025-02-09,2025-02-01,2022-10-29,2022-10-23,2022-10-01,838,4. 01-03 (infrequent),NaN,0.556978


In [35]:
loyalty_agg = loyalty_data[['dt', 'loyalty_segment', 'user_id']][loyalty_data.loyalty_segment!='0.0 (new install)'].copy()

loyalty_agg = loyalty_agg.groupby(['dt','loyalty_segment']).agg(
    unique_users = ('user_id', 'nunique'),
).reset_index().sort_values(by=['dt','loyalty_segment'])

# Add total unique users per day
loyalty_agg['total_users_per_day'] = loyalty_agg.groupby('dt')['unique_users'].transform('sum')

# Add percentage column
loyalty_agg['percentage'] = (loyalty_agg['unique_users'] / loyalty_agg['total_users_per_day'])

loyalty_agg

,dt,loyalty_segment,unique_users,total_users_per_day,percentage
0,2025-01-01,1. 26-28 (dedicated),88524,156764,0.564696
1,2025-01-01,2. 19-25 (frequent),37491,156764,0.239156
2,2025-01-01,3. 04-18 (moderate),25878,156764,0.165076
3,2025-01-01,4. 01-03 (infrequent),4871,156764,0.031072
4,2025-01-02,1. 26-28 (dedicated),89029,161032,0.552865
...,...,...,...,...,...
2295,2026-07-28,4. 01-03 (infrequent),2357,64698,0.036431
2296,2026-07-29,1. 26-28 (dedicated),37184,64443,0.577006
2297,2026-07-29,2. 19-25 (frequent),14207,64443,0.220458
2298,2026-07-29,3. 04-18 (moderate),10583,64443,0.164223


In [36]:
# Create a line chart showing return rates over time
fig = px.line(
    loyalty_agg,
    x='dt',
    y=['percentage'],
    color='loyalty_segment',
    title='Loyalty Segment Unique Users Over Time',
    labels={'dt': 'Date', 'value': 'Unique Users', 'variable': 'Loyalty Segment'},
    markers=False,
    height=600,
    width=1500
)

fig.show()

## Loyalty Segment Forecast & Scenario Analysis

Same 6-step pipeline as the RRW section above (same `aux_functions.py` helpers), applied to a loyalty segment's daily share of DAU. Change `segment_value` to retarget any other segment (e.g. `'2. 19-25 (frequent)'`, `'3. 04-18 (moderate)'`, `'4. 01-03 (infrequent)'`).

## 1. Data prep + trend/seasonality decomposition

### Parameters

In [37]:
# --- All tunable parameters for this section, in one place ---

segment_value = '1. 26-28 (dedicated)'  # which loyalty segment to analyze — swap to retarget (e.g. '2. 19-25 (frequent)')

outlier_threshold = 3.5                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct_loyalty = 0.005              # Sec 6: relative uplift applied to the forecast curve (e.g. 0.05 = +5%)

In [38]:
# hide-output
loyalty_seg = loyalty_agg[loyalty_agg['loyalty_segment'] == segment_value][['dt', 'percentage']].dropna().copy()
loyalty_seg['dt'] = pd.to_datetime(loyalty_seg['dt'])
loyalty_seg = loyalty_seg.set_index('dt').asfreq('D')
n_gaps = loyalty_seg['percentage'].isna().sum()
loyalty_seg['percentage'] = loyalty_seg['percentage'].interpolate()  # fill any single-day calendar gaps

loyalty_series = loyalty_seg['percentage']
loyalty_series.name = segment_value

print(f"{segment_value}: {len(loyalty_seg)} days, {loyalty_seg.index.min().date()} -> {loyalty_seg.index.max().date()} ({n_gaps} interpolated gap-days)")

1. 26-28 (dedicated): 575 days, 2025-01-01 -> 2026-07-29 (0 interpolated gap-days)


In [39]:
# hide-output
loyalty_decomp, loyalty_adf_stat, loyalty_adf_p = decompose_series(loyalty_series, period=7)
print(f"ADF test on STL residual: stat={loyalty_adf_stat:.3f}, p={loyalty_adf_p:.4f} -> {'stationary' if loyalty_adf_p < 0.05 else 'NOT stationary'}")

ADF test on STL residual: stat=-10.530, p=0.0000 -> stationary


In [40]:
# Chart — STL decomposition: observed+trend / weekly seasonal / residual
plot_stl_decomposition(loyalty_decomp, title=f'{segment_value} — STL decomposition (weekly seasonality)').show()

## 2. Variance & outliers

Same MAD-based robust z-score as the RRW section (see the plain-language explanation there for how it works).

In [41]:
# hide-output
loyalty_decomp['modified_z'], loyalty_decomp['is_outlier'] = flag_robust_outliers(loyalty_decomp['resid'], threshold=outlier_threshold)

cov = loyalty_decomp['resid'].std() / loyalty_decomp['value'].mean()
print(f"Residual std: {loyalty_decomp['resid'].std():.5f}  |  MAD: {(loyalty_decomp['resid'] - loyalty_decomp['resid'].median()).abs().median():.5f}  |  series mean: {loyalty_decomp['value'].mean():.4f}")
print(f"Coefficient of variation (resid std / series mean): {cov:.3%}")
print(f"Outliers flagged (|modified z| > {outlier_threshold}): {loyalty_decomp['is_outlier'].sum()} / {len(loyalty_decomp)} days")

loyalty_decomp[loyalty_decomp['is_outlier']][['value', 'resid', 'modified_z']]

Residual std: 0.00305  |  MAD: 0.00099  |  series mean: 0.5730
Coefficient of variation (resid std / series mean): 0.533%
Outliers flagged (|modified z| > 3.5): 46 / 575 days


,value,resid,modified_z
dt,,,
2025-01-08,0.554319,-0.010587,-7.245594
2025-01-15,0.556990,-0.008792,-6.020333
2025-02-17,0.578370,-0.006128,-4.201803
2025-02-25,0.591889,0.005680,3.858505
2025-02-26,0.595783,0.007938,5.399275
2025-02-27,0.596321,0.006468,4.396520
2025-05-02,0.578721,0.005317,3.610490
2025-05-23,0.573776,0.009220,6.274810
2025-05-24,0.575560,0.010712,7.293331


In [42]:
# Chart — observed series with flagged outlier days marked
plot_outliers(loyalty_decomp, loyalty_decomp['is_outlier'], title=f'{segment_value} with robust-outlier days flagged').show()

## 3. Change-point detection (regime shifts)

Same PELT-based unsupervised detection as the RRW section above, on this segment's deseasonalized share.

In [43]:
# hide-output
changepoint_dates_loyalty, changepoint_penalty_loyalty = detect_changepoints(loyalty_decomp['deseasonalized'], min_segment_days=min_segment_days)

print(f"Penalty: {changepoint_penalty_loyalty:.2f}  |  Change-points detected: {len(changepoint_dates_loyalty)}")
for d in changepoint_dates_loyalty:
    print(' ', d.date())

Penalty: 6.35  |  Change-points detected: 8
  2025-01-25
  2025-04-10
  2025-05-25
  2025-06-19
  2025-12-11
  2026-01-25
  2026-02-24
  2026-06-04


In [44]:
# Chart — observed series with detected change-points marked
plot_changepoints(loyalty_decomp, changepoint_dates_loyalty, title=f'{segment_value} with detected regime changes (dashed lines)').show()

## 4. Forecast (3 months ahead)

Same training-window walk-back + logit-space SARIMAX approach as the RRW section above.

In [45]:
# hide-output
# Walk back through detected regimes until we have >= min_training_days (set in Parameters above)
training_start_loyalty = select_training_start(loyalty_decomp.index, changepoint_dates_loyalty, min_training_days=min_training_days)
train_loyalty = loyalty_series.loc[training_start_loyalty:]
print(f"Training window: {training_start_loyalty.date()} -> {loyalty_decomp.index.max().date()} ({len(train_loyalty)} days)")

Training window: 2026-02-24 -> 2026-07-29 (156 days)


In [46]:
# hide-output
backtest_mae_loyalty, backtest_mape_loyalty, forecast_df_loyalty = fit_rate_forecast(
    train_loyalty, order=sarimax_order, seasonal_order=sarimax_seasonal_order,
    horizon=forecast_horizon_days, backtest_days=backtest_days, ci_alpha=ci_alpha,
)
print(f"Backtest (last {backtest_days} days of training window): MAE={backtest_mae_loyalty:.5f}  MAPE={backtest_mape_loyalty:.3%}")

Backtest (last 28 days of training window): MAE=0.00304  MAPE=0.527%


In [47]:
# hide-output
print(f"Forecast horizon: {forecast_df_loyalty.index.min().date()} -> {forecast_df_loyalty.index.max().date()}")
for h in [30, 60, 90]:
    row = forecast_df_loyalty.iloc[h - 1]
    print(f"  +{h}d ({forecast_df_loyalty.index[h-1].date()}): {row['forecast']:.4f}  [{row['low']:.4f}, {row['high']:.4f}]")

Forecast horizon: 2026-07-30 -> 2026-10-27
  +30d (2026-08-28): 0.5820  [0.5542, 0.6092]
  +60d (2026-09-27): 0.5716  [0.5186, 0.6230]
  +90d (2026-10-27): 0.5677  [0.4859, 0.6460]


## 5. Summary: history + forecast

In [48]:
# Chart — training window history + 90-day forecast with 80% band
plot_forecast_summary(
    loyalty_series, train_loyalty, forecast_df_loyalty, training_start_loyalty,
    title=f'{segment_value} — history + {forecast_horizon_days}-day forecast', ci_alpha=ci_alpha,
).show()

In [49]:
# Summary table — forecast at key checkpoints
summary_table_loyalty = forecast_df_loyalty.iloc[[29, 59, 89]].reset_index().rename(columns={
    'dt': 'Date', 'forecast': 'P50 (forecast)', 'low': 'P10', 'high': 'P90',
})
summary_table_loyalty.insert(0, 'Days out', [30, 60, 90])
summary_table_loyalty[['P10', 'P50 (forecast)', 'P90']] = summary_table_loyalty[['P10', 'P50 (forecast)', 'P90']].round(4)
summary_table_loyalty

,Days out,Date,P50 (forecast),P10,P90
0,30,2026-08-28,0.5820,0.5542,0.6092
1,60,2026-09-27,0.5716,0.5186,0.6230
2,90,2026-10-27,0.5677,0.4859,0.6460


## 6. Scenario: what if this segment's share gets a % uplift

Same standalone overlay approach as the RRW section. Adjust `uplift_pct_loyalty` to try different scenarios.

In [50]:
# hide-output
scenario_df_loyalty = apply_uplift_scenario(forecast_df_loyalty, uplift_pct_loyalty)

print(f"Uplift scenario: {uplift_pct_loyalty:+.1%} relative")
for h in [30, 60, 90]:
    base = scenario_df_loyalty['forecast'].iloc[h - 1]
    up = scenario_df_loyalty['forecast_uplift'].iloc[h - 1]
    print(f"  +{h}d: baseline={base:.4f}  uplifted={up:.4f}  (delta={up - base:+.4f})")

Uplift scenario: +0.5% relative
  +30d: baseline=0.5820  uplifted=0.5849  (delta=+0.0029)
  +60d: baseline=0.5716  uplifted=0.5744  (delta=+0.0029)
  +90d: baseline=0.5677  uplifted=0.5706  (delta=+0.0028)


In [51]:
# Chart — baseline forecast vs uplift scenario
plot_uplift_scenario(
    loyalty_series, scenario_df_loyalty, uplift_pct_loyalty,
    title=f'{segment_value} — baseline vs {uplift_pct_loyalty:+.1%} uplift scenario', lookback_days=60,
).show()

## 7. Period comparison: Period 1 vs Period 2

Average `segment_value`'s share within each of the two shared periods defined at the top of the notebook, and the relative uplift between them.

In [52]:
# hide-output
loyalty_period_comparison = compare_periods(loyalty_series, period1_start, period1_end, period2_start, period2_end)

print(f"Period 1 ({period1_start} -> {period1_end}, n={loyalty_period_comparison['period1_n']} days): avg share = {loyalty_period_comparison['period1_mean']:.4f}")
print(f"Period 2 ({period2_start} -> {period2_end}, n={loyalty_period_comparison['period2_n']} days): avg share = {loyalty_period_comparison['period2_mean']:.4f}")
print(f"Uplift: {loyalty_period_comparison['abs_diff']:+.4f}  ({loyalty_period_comparison['pct_uplift']:+.1%} relative)")

Period 1 (2025-07-01 -> 2025-09-30, n=92 days): avg share = 0.5621
Period 2 (2026-01-01 -> 2026-06-30, n=181 days): avg share = 0.5868
Uplift: +0.0247  (+4.4% relative)


In [53]:
# Chart — Period 1 vs Period 2 average
plot_period_comparison(
    loyalty_period_comparison, title=f'{segment_value}: Period 1 vs Period 2 average', y_label=segment_value,
).show()

# ARPDAU

In [98]:
# --- All tunable parameters for this section, in one place ---

segment_value = '1. 26-28 (dedicated)'  # which loyalty segment to analyze — swap to retarget (e.g. '2. 19-25 (frequent)')

outlier_threshold = 7                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct_loyalty = 0.02              # Sec 6: relative uplift applied to the forecast curve (e.g. 0.02 = +2%)

In [99]:
loyalty_data

,user_id,dt,dt_week,dt_month,install_dt,install_dt_week,install_dt_month,days_since_install,loyalty_segment,usd_net_iap_revenue,usd_net_ad_revenue
0,93752F4F22B5A9CD,2025-10-15,2025-10-12,2025-10-01,2024-12-18,2024-12-15,2024-12-01,301,1. 26-28 (dedicated),NaN,0.123387
1,FB28110522A079A2,2025-03-23,2025-03-23,2025-03-01,2023-09-09,2023-09-03,2023-09-01,561,3. 04-18 (moderate),NaN,0.016943
2,7FB880617AC04268,2025-10-12,2025-10-12,2025-10-01,2024-04-13,2024-04-07,2024-04-01,547,1. 26-28 (dedicated),NaN,0.112009
3,CB7C0252FE5AC78A,2025-04-26,2025-04-20,2025-04-01,2024-10-07,2024-10-06,2024-10-01,201,1. 26-28 (dedicated),NaN,0.139790
4,55C99BE1E3255D0A,2025-05-24,2025-05-18,2025-05-01,2022-11-10,2022-11-06,2022-11-01,926,3. 04-18 (moderate),NaN,0.009639
...,...,...,...,...,...,...,...,...,...,...,...
73885672,9A104359CBCC77B1,2025-12-07,2025-12-07,2025-12-01,2024-05-28,2024-05-26,2024-05-01,558,1. 26-28 (dedicated),NaN,0.053933
73885673,E001C64794784601,2026-02-10,2026-02-08,2026-02-01,2022-03-29,2022-03-27,2022-03-01,1414,1. 26-28 (dedicated),NaN,NaN
73885674,BD5B39F6914601B4,2025-09-16,2025-09-14,2025-09-01,2023-05-06,2023-04-30,2023-05-01,864,3. 04-18 (moderate),12.414838,NaN
73885675,90B05908B794DB7F,2025-02-13,2025-02-09,2025-02-01,2022-10-29,2022-10-23,2022-10-01,838,4. 01-03 (infrequent),NaN,0.556978


In [100]:
arpdau_agg = loyalty_data[['dt', 'user_id', 'usd_net_iap_revenue','usd_net_ad_revenue']].copy()

arpdau_agg = arpdau_agg.groupby(['dt']).agg(
    unique_users = ('user_id', pd.Series.nunique),
    total_iap_revenue = ('usd_net_iap_revenue', 'sum'),
    total_ad_revenue = ('usd_net_ad_revenue', 'sum'),
).reset_index().sort_values(by=['dt'])

arpdau_agg['arpdau_iap'] = (arpdau_agg['total_iap_revenue'] / arpdau_agg['unique_users']).round(4)
arpdau_agg['arpdau_ad'] = (arpdau_agg['total_ad_revenue'] / arpdau_agg['unique_users']).round(4)

arpdau_agg

,dt,unique_users,total_iap_revenue,total_ad_revenue,arpdau_iap,arpdau_ad
0,2025-01-01,184241,38727.305724,15588.426723,0.2102,0.0846
1,2025-01-02,189595,35609.032592,15305.733612,0.1878,0.0807
2,2025-01-03,189066,38767.160679,15391.555793,0.2050,0.0814
3,2025-01-04,188451,42423.292575,16571.649227,0.2251,0.0879
4,2025-01-05,191441,37484.262951,17123.758805,0.1958,0.0894
...,...,...,...,...,...,...
570,2026-07-25,68945,24113.474799,5956.283819,0.3497,0.0864
571,2026-07-26,70609,19226.551819,6592.324334,0.2723,0.0934
572,2026-07-27,70510,16457.642862,6149.254661,0.2334,0.0872
573,2026-07-28,69741,15347.176843,5948.942201,0.2201,0.0853


In [101]:
arpdau_seg_agg = loyalty_data[['dt', 'loyalty_segment', 'user_id', 'usd_net_iap_revenue','usd_net_ad_revenue']].copy()

arpdau_seg_agg = arpdau_seg_agg.groupby(['dt','loyalty_segment']).agg(
    unique_users = ('user_id', 'nunique'),
    total_iap_revenue = ('usd_net_iap_revenue', 'sum'),
    total_ad_revenue = ('usd_net_ad_revenue', 'sum'),
).reset_index().sort_values(by=['dt','loyalty_segment'])

arpdau_seg_agg['total_users_per_day'] = arpdau_seg_agg.groupby('dt')['unique_users'].transform('sum')

arpdau_seg_agg['arpdau_local_iap'] = arpdau_seg_agg['total_iap_revenue'] / arpdau_seg_agg['unique_users']
arpdau_seg_agg['arpdau_local_ad'] = arpdau_seg_agg['total_ad_revenue'] / arpdau_seg_agg['unique_users']

arpdau_seg_agg['arpdau_iap'] = arpdau_seg_agg['total_iap_revenue'] / arpdau_seg_agg['total_users_per_day']
arpdau_seg_agg['arpdau_ad'] = arpdau_seg_agg['total_ad_revenue'] / arpdau_seg_agg['total_users_per_day']

arpdau_seg_agg

,dt,loyalty_segment,unique_users,total_iap_revenue,total_ad_revenue,total_users_per_day,arpdau_local_iap,arpdau_local_ad,arpdau_iap,arpdau_ad
0,2025-01-01,0.0 (new install),27477,7727.672059,2229.679960,184241,0.281241,0.081147,0.041943,0.012102
1,2025-01-01,1. 26-28 (dedicated),88524,21573.657445,8381.290089,184241,0.243704,0.094678,0.117095,0.045491
2,2025-01-01,2. 19-25 (frequent),37491,5124.186204,2657.692628,184241,0.136678,0.070889,0.027812,0.014425
3,2025-01-01,3. 04-18 (moderate),25878,3706.306264,1931.501006,184241,0.143222,0.074639,0.020117,0.010484
4,2025-01-01,4. 01-03 (infrequent),4871,595.483753,388.263040,184241,0.122251,0.079709,0.003232,0.002107
...,...,...,...,...,...,...,...,...,...,...
2870,2026-07-29,0.0 (new install),4884,811.083995,487.219256,69327,0.166070,0.099758,0.011699,0.007028
2871,2026-07-29,1. 26-28 (dedicated),37184,11125.201331,3450.324881,69327,0.299193,0.092791,0.160474,0.049769
2872,2026-07-29,2. 19-25 (frequent),14207,2503.429393,972.477028,69327,0.176211,0.068451,0.036110,0.014027
2873,2026-07-29,3. 04-18 (moderate),10583,1872.794877,897.475110,69327,0.176963,0.084803,0.027014,0.012946


In [102]:
# Create a line chart showing return rates over time
fig = px.line(
    arpdau_agg,
    x='dt',
    y=['arpdau_iap', 'arpdau_ad'],
    title='ARPDAU Over Time',
    labels={'dt': 'Date', 'value': 'ARPDAU', 'variable': 'Revenue Type'},
    markers=False,
    height=600,
    width=1500,
    hover_data={'unique_users': True, 'total_iap_revenue': ':.0f', 'total_ad_revenue': ':.0f'},
)

fig.show()

In [103]:
# Create a line chart showing return rates over time
fig = px.line(
    arpdau_seg_agg[arpdau_seg_agg['loyalty_segment'] == segment_value],
    x='dt',
    y=['arpdau_local_iap', 'arpdau_local_ad', 'arpdau_iap', 'arpdau_ad'],
    title='ARPDAU Over Time by Loyalty Segment',
    labels={'dt': 'Date', 'value': 'ARPDAU', 'variable': 'Loyalty Segment'},
    markers=False,
    height=600,
    width=1500
)

fig.show()

## Core ARPDAU (excluding new installs)

New installs are about to slow down significantly, and day-0 users are close to guaranteed $0 revenue — so as they shrink as a share of DAU, blended ARPDAU will rise mechanically even if nothing about how well existing users are monetized has changed (a mix-shift artifact, not a real improvement). **Core ARPDAU** re-blends revenue and users across segments 1-4 only (excluding `0.0 (new install)`), so it isn't affected by UA volume changes. The 7-section pipelines below run on core ARPDAU; blended ARPDAU (all segments, computed earlier) is kept as a reference chart so any future divergence between the two is itself a diagnostic signal.

In [104]:
# hide-output
from aux_functions import fit_value_forecast

In [105]:
# hide-output
core_arpdau_agg = loyalty_data[loyalty_data['loyalty_segment'] != '0.0 (new install)'][['dt', 'user_id', 'usd_net_iap_revenue', 'usd_net_ad_revenue']].copy()

core_arpdau_agg = core_arpdau_agg.groupby('dt').agg(
    unique_users=('user_id', 'nunique'),
    total_iap_revenue=('usd_net_iap_revenue', 'sum'),
    total_ad_revenue=('usd_net_ad_revenue', 'sum'),
).reset_index().sort_values('dt')
core_arpdau_agg['dt'] = pd.to_datetime(core_arpdau_agg['dt'])

core_arpdau_agg['arpdau_iap'] = core_arpdau_agg['total_iap_revenue'] / core_arpdau_agg['unique_users']
core_arpdau_agg['arpdau_ad'] = core_arpdau_agg['total_ad_revenue'] / core_arpdau_agg['unique_users']

core_arpdau_agg

,dt,unique_users,total_iap_revenue,total_ad_revenue,arpdau_iap,arpdau_ad
0,2025-01-01,156764,30999.633665,13358.746763,0.197747,0.085216
1,2025-01-02,161032,27956.151133,13103.455542,0.173606,0.081372
2,2025-01-03,161128,30452.024698,13119.177393,0.188993,0.081421
3,2025-01-04,160749,33677.306935,14102.549706,0.209502,0.087730
4,2025-01-05,163222,28510.395094,14533.704699,0.174673,0.089043
...,...,...,...,...,...,...
570,2026-07-25,63470,23210.101836,5465.526336,0.365686,0.086112
571,2026-07-26,65046,18170.636506,6088.322580,0.279351,0.093600
572,2026-07-27,65095,15399.815964,5683.904655,0.236574,0.087317
573,2026-07-28,64698,14489.480190,5465.857762,0.223956,0.084483


In [142]:
# Chart — blended (all segments) vs core (excl. new installs) ARPDAU IAP
compare_df_iap = arpdau_agg[['dt', 'arpdau_iap']].copy()
compare_df_iap['dt'] = pd.to_datetime(compare_df_iap['dt'])
compare_df_iap = compare_df_iap.merge(core_arpdau_agg[['dt', 'arpdau_iap']], on='dt', suffixes=('_blended', '_core'))

fig = px.line(
    compare_df_iap, x='dt', y=['arpdau_iap_blended', 'arpdau_iap_core'],
    title='ARPDAU IAP: blended (all segments) vs core (excl. new installs)',
    labels={'dt': 'Date', 'value': 'ARPDAU (IAP)', 'variable': ''},
    width=1400, height=500,
)
fig.show()

# Chart — blended (all segments) vs core (excl. new installs) ARPDAU Ad
compare_df_ad = arpdau_agg[['dt', 'arpdau_ad']].copy()
compare_df_ad['dt'] = pd.to_datetime(compare_df_ad['dt'])
compare_df_ad = compare_df_ad.merge(core_arpdau_agg[['dt', 'arpdau_ad']], on='dt', suffixes=('_blended', '_core'))

fig = px.line(
    compare_df_ad, x='dt', y=['arpdau_ad_blended', 'arpdau_ad_core'],
    title='ARPDAU Ad: blended (all segments) vs core (excl. new installs)',
    labels={'dt': 'Date', 'value': 'ARPDAU (Ad)', 'variable': ''},
    width=1400, height=500,
)
fig.show()

## ARPDAU IAP (Core) Forecast & Scenario Analysis

Same 7-step pipeline as the RRW/Loyalty sections above, applied to core ARPDAU (IAP revenue per DAU, excluding new installs). Two differences from those sections: **Section 4 uses a log-transform** (`fit_value_forecast`) instead of the logit-transform used for rate metrics, since ARPDAU is an unbounded positive value, not a [0,1] proportion — and **Section 6's uplift is unclipped** for the same reason (revenue has no natural 100% ceiling).

### 1. Data prep + trend/seasonality decomposition

In [108]:
# --- All tunable parameters for this section, in one place ---

metric_col = 'arpdau_iap'            # which core ARPDAU series to analyze

outlier_threshold = 7                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct_arpdau_iap = 0.02    # Sec 6: relative uplift applied to the forecast curve (e.g. 0.02 = +2%)

In [109]:
# hide-output
arpdau_iap_series = core_arpdau_agg.set_index('dt')[metric_col].asfreq('D')
n_gaps = arpdau_iap_series.isna().sum()
arpdau_iap_series = arpdau_iap_series.interpolate()  # fill any single-day calendar gaps
arpdau_iap_series.name = metric_col

print(f"{metric_col}: {len(arpdau_iap_series)} days, {arpdau_iap_series.index.min().date()} -> {arpdau_iap_series.index.max().date()} ({n_gaps} interpolated gap-days)")

arpdau_iap: 575 days, 2025-01-01 -> 2026-07-29 (0 interpolated gap-days)


In [110]:
# hide-output
arpdau_iap_decomp, arpdau_iap_adf_stat, arpdau_iap_adf_p = decompose_series(arpdau_iap_series, period=7)
print(f"ADF test on STL residual: stat={arpdau_iap_adf_stat:.3f}, p={arpdau_iap_adf_p:.4f} -> {'stationary' if arpdau_iap_adf_p < 0.05 else 'NOT stationary'}")

ADF test on STL residual: stat=-8.993, p=0.0000 -> stationary


In [111]:
# Chart — STL decomposition: observed+trend / weekly seasonal / residual
plot_stl_decomposition(arpdau_iap_decomp, title=f'{metric_col} (core) — STL decomposition (weekly seasonality)').show()

### 2. Variance & outliers

Same MAD-based robust z-score as the RRW section (see the plain-language explanation there for how it works).

In [112]:
# hide-output
arpdau_iap_decomp['modified_z'], arpdau_iap_decomp['is_outlier'] = flag_robust_outliers(arpdau_iap_decomp['resid'], threshold=outlier_threshold)

cov = arpdau_iap_decomp['resid'].std() / arpdau_iap_decomp['value'].mean()
print(f"Residual std: {arpdau_iap_decomp['resid'].std():.5f}  |  MAD: {(arpdau_iap_decomp['resid'] - arpdau_iap_decomp['resid'].median()).abs().median():.5f}  |  series mean: {arpdau_iap_decomp['value'].mean():.4f}")
print(f"Coefficient of variation (resid std / series mean): {cov:.3%}")
print(f"Outliers flagged (|modified z| > {outlier_threshold}): {arpdau_iap_decomp['is_outlier'].sum()} / {len(arpdau_iap_decomp)} days")

arpdau_iap_decomp[arpdau_iap_decomp['is_outlier']][['value', 'resid', 'modified_z']]

Residual std: 0.06329  |  MAD: 0.00573  |  series mean: 0.2404
Coefficient of variation (resid std / series mean): 26.327%
Outliers flagged (|modified z| > 7): 82 / 575 days


,value,resid,modified_z
dt,,,
2025-01-17,0.341585,0.150421,17.708535
2025-01-24,0.498019,0.302598,35.623913
2025-01-25,0.296291,0.094189,11.088462
2025-01-31,0.187818,-0.160066,-18.844228
2025-02-07,0.180053,-0.193214,-22.746604
...,...,...,...
2026-07-05,0.390762,0.109498,12.890782
2026-07-10,0.221544,-0.147751,-17.394431
2026-07-11,0.224435,-0.101732,-11.976748


In [113]:
# Chart — observed series with flagged outlier days marked
plot_outliers(arpdau_iap_decomp, arpdau_iap_decomp['is_outlier'], title=f'{metric_col} (core) with robust-outlier days flagged').show()

### 3. Change-point detection (regime shifts)

Same PELT-based unsupervised detection as the RRW section above, on this metric's deseasonalized series.

In [114]:
# hide-output
arpdau_iap_changepoint_dates, arpdau_iap_changepoint_penalty = detect_changepoints(arpdau_iap_decomp['deseasonalized'], min_segment_days=min_segment_days)

print(f"Penalty: {arpdau_iap_changepoint_penalty:.2f}  |  Change-points detected: {len(arpdau_iap_changepoint_dates)}")
for d in arpdau_iap_changepoint_dates:
    print(' ', d.date())

Penalty: 6.35  |  Change-points detected: 4
  2025-05-15
  2025-08-03
  2025-12-06
  2026-03-26


In [115]:
# Chart — observed series with detected change-points marked
plot_changepoints(arpdau_iap_decomp, arpdau_iap_changepoint_dates, title=f'{metric_col} (core) with detected regime changes (dashed lines)').show()

### 4. Forecast (3 months ahead)

Same training-window walk-back as the RRW section, but the SARIMAX fit uses a **log-transform** (`fit_value_forecast`) instead of logit, since ARPDAU is an unbounded positive value, not a [0,1] rate. Revenue metrics are inherently noisier day-to-day than rate metrics (lumpier, driven by a smaller number of paying/high-value users), so expect a wider backtest error than the RRW/Loyalty sections.

In [116]:
# hide-output
# Walk back through detected regimes until we have >= min_training_days (set in Parameters above)
arpdau_iap_training_start = select_training_start(arpdau_iap_decomp.index, arpdau_iap_changepoint_dates, min_training_days=min_training_days)
arpdau_iap_train = arpdau_iap_series.loc[arpdau_iap_training_start:]
print(f"Training window: {arpdau_iap_training_start.date()} -> {arpdau_iap_decomp.index.max().date()} ({len(arpdau_iap_train)} days)")

Training window: 2026-03-26 -> 2026-07-29 (126 days)


In [117]:
# hide-output
arpdau_iap_backtest_mae, arpdau_iap_backtest_mape, arpdau_iap_forecast_df = fit_value_forecast(
    arpdau_iap_train, order=sarimax_order, seasonal_order=sarimax_seasonal_order,
    horizon=forecast_horizon_days, backtest_days=backtest_days, ci_alpha=ci_alpha,
)
print(f"Backtest (last {backtest_days} days of training window): MAE={arpdau_iap_backtest_mae:.5f}  MAPE={arpdau_iap_backtest_mape:.3%}")

Backtest (last 28 days of training window): MAE=0.04501  MAPE=16.788%


In [118]:
# hide-output
print(f"Forecast horizon: {arpdau_iap_forecast_df.index.min().date()} -> {arpdau_iap_forecast_df.index.max().date()}")
for h in [30, 60, 90]:
    row = arpdau_iap_forecast_df.iloc[h - 1]
    print(f"  +{h}d ({arpdau_iap_forecast_df.index[h-1].date()}): {row['forecast']:.4f}  [{row['low']:.4f}, {row['high']:.4f}]")

Forecast horizon: 2026-07-30 -> 2026-10-27
  +30d (2026-08-28): 0.3323  [0.2459, 0.4491]
  +60d (2026-09-27): 0.2786  [0.2038, 0.3809]
  +90d (2026-10-27): 0.2112  [0.1524, 0.2928]


### 5. Summary: history + forecast

In [119]:
# Chart — training window history + 90-day forecast with 80% band
plot_forecast_summary(
    arpdau_iap_series, arpdau_iap_train, arpdau_iap_forecast_df, arpdau_iap_training_start,
    title=f'{metric_col} (core) — history + {forecast_horizon_days}-day forecast', ci_alpha=ci_alpha,
).show()

In [120]:
# Summary table — forecast at key checkpoints
arpdau_iap_summary_table = arpdau_iap_forecast_df.iloc[[29, 59, 89]].reset_index().rename(columns={
    'dt': 'Date', 'forecast': 'P50 (forecast)', 'low': 'P10', 'high': 'P90',
})
arpdau_iap_summary_table.insert(0, 'Days out', [30, 60, 90])
arpdau_iap_summary_table[['P10', 'P50 (forecast)', 'P90']] = arpdau_iap_summary_table[['P10', 'P50 (forecast)', 'P90']].round(4)
arpdau_iap_summary_table

,Days out,Date,P50 (forecast),P10,P90
0,30,2026-08-28,0.3323,0.2459,0.4491
1,60,2026-09-27,0.2786,0.2038,0.3809
2,90,2026-10-27,0.2112,0.1524,0.2928


### 6. Scenario: what if core ARPDAU IAP gets a % uplift

Same standalone overlay approach as the RRW section — no upper clip here, since revenue has no natural 100% ceiling. Adjust `uplift_pct_arpdau_iap` to try different scenarios.

In [121]:
# hide-output
arpdau_iap_scenario_df = apply_uplift_scenario(arpdau_iap_forecast_df, uplift_pct_arpdau_iap, clip_upper=None)

print(f"Uplift scenario: {uplift_pct_arpdau_iap:+.1%} relative")
for h in [30, 60, 90]:
    base = arpdau_iap_scenario_df['forecast'].iloc[h - 1]
    up = arpdau_iap_scenario_df['forecast_uplift'].iloc[h - 1]
    print(f"  +{h}d: baseline={base:.4f}  uplifted={up:.4f}  (delta={up - base:+.4f})")

Uplift scenario: +2.0% relative
  +30d: baseline=0.3323  uplifted=0.3390  (delta=+0.0066)
  +60d: baseline=0.2786  uplifted=0.2842  (delta=+0.0056)
  +90d: baseline=0.2112  uplifted=0.2154  (delta=+0.0042)


In [122]:
# Chart — baseline forecast vs uplift scenario
plot_uplift_scenario(
    arpdau_iap_series, arpdau_iap_scenario_df, uplift_pct_arpdau_iap,
    title=f'{metric_col} (core) — baseline vs {uplift_pct_arpdau_iap:+.1%} uplift scenario', lookback_days=60,
).show()

### 7. Period comparison: Period 1 vs Period 2

Average core `metric_col` within each of the two shared periods defined at the top of the notebook, and the relative uplift between them.

In [123]:
# hide-output
arpdau_iap_period_comparison = compare_periods(arpdau_iap_series, period1_start, period1_end, period2_start, period2_end)

print(f"Period 1 ({period1_start} -> {period1_end}, n={arpdau_iap_period_comparison['period1_n']} days): avg {metric_col} = {arpdau_iap_period_comparison['period1_mean']:.4f}")
print(f"Period 2 ({period2_start} -> {period2_end}, n={arpdau_iap_period_comparison['period2_n']} days): avg {metric_col} = {arpdau_iap_period_comparison['period2_mean']:.4f}")
print(f"Uplift: {arpdau_iap_period_comparison['abs_diff']:+.4f}  ({arpdau_iap_period_comparison['pct_uplift']:+.1%} relative)")

Period 1 (2025-07-01 -> 2025-09-30, n=92 days): avg arpdau_iap = 0.2327
Period 2 (2026-01-01 -> 2026-06-30, n=181 days): avg arpdau_iap = 0.2740
Uplift: +0.0413  (+17.7% relative)


In [124]:
# Chart — Period 1 vs Period 2 average
plot_period_comparison(
    arpdau_iap_period_comparison, title=f'{metric_col} (core): Period 1 vs Period 2 average', y_label=metric_col,
).show()

## ARPDAU Ad (Core) Forecast & Scenario Analysis

Same 7-step pipeline as the RRW/Loyalty sections above, applied to core ARPDAU (Ad revenue per DAU, excluding new installs). Two differences from those sections: **Section 4 uses a log-transform** (`fit_value_forecast`) instead of the logit-transform used for rate metrics, since ARPDAU is an unbounded positive value, not a [0,1] proportion — and **Section 6's uplift is unclipped** for the same reason (revenue has no natural 100% ceiling).

### 1. Data prep + trend/seasonality decomposition

In [143]:
# --- All tunable parameters for this section, in one place ---

metric_col = 'arpdau_ad'            # which core ARPDAU series to analyze

outlier_threshold = 7                # Sec 2: modified z-score threshold for flagging outliers (Iglewicz & Hoaglin standard)
min_segment_days = 14                  # Sec 3: minimum days between two detected change-points (regimes)
min_training_days = 90                 # Sec 4: forecast training window — walk back through regimes until >= this many days
sarimax_order = (1, 1, 1)              # Sec 4: SARIMAX (p, d, q) non-seasonal order
sarimax_seasonal_order = (1, 1, 1, 7)  # Sec 4: SARIMAX seasonal order, period=7 (weekly)
forecast_horizon_days = 90             # Sec 4: how many days ahead to forecast
backtest_days = 28                     # Sec 4: holdout days for the pre-forecast backtest
ci_alpha = 0.20                        # Sec 4: forecast interval width (0.20 -> 80% interval, ~P10/P90)
uplift_pct_arpdau_ad = 0.02    # Sec 6: relative uplift applied to the forecast curve (e.g. 0.02 = +2%)

In [144]:
# hide-output
arpdau_ad_series = core_arpdau_agg.set_index('dt')[metric_col].asfreq('D')
n_gaps = arpdau_ad_series.isna().sum()
arpdau_ad_series = arpdau_ad_series.interpolate()  # fill any single-day calendar gaps
arpdau_ad_series.name = metric_col

print(f"{metric_col}: {len(arpdau_ad_series)} days, {arpdau_ad_series.index.min().date()} -> {arpdau_ad_series.index.max().date()} ({n_gaps} interpolated gap-days)")

arpdau_ad: 575 days, 2025-01-01 -> 2026-07-29 (0 interpolated gap-days)


In [145]:
# hide-output
arpdau_ad_decomp, arpdau_ad_adf_stat, arpdau_ad_adf_p = decompose_series(arpdau_ad_series, period=7)
print(f"ADF test on STL residual: stat={arpdau_ad_adf_stat:.3f}, p={arpdau_ad_adf_p:.4f} -> {'stationary' if arpdau_ad_adf_p < 0.05 else 'NOT stationary'}")

ADF test on STL residual: stat=-19.399, p=0.0000 -> stationary


In [146]:
# Chart — STL decomposition: observed+trend / weekly seasonal / residual
plot_stl_decomposition(arpdau_ad_decomp, title=f'{metric_col} (core) — STL decomposition (weekly seasonality)').show()

### 2. Variance & outliers

Same MAD-based robust z-score as the RRW section (see the plain-language explanation there for how it works).

In [147]:
# hide-output
arpdau_ad_decomp['modified_z'], arpdau_ad_decomp['is_outlier'] = flag_robust_outliers(arpdau_ad_decomp['resid'], threshold=outlier_threshold)

cov = arpdau_ad_decomp['resid'].std() / arpdau_ad_decomp['value'].mean()
print(f"Residual std: {arpdau_ad_decomp['resid'].std():.5f}  |  MAD: {(arpdau_ad_decomp['resid'] - arpdau_ad_decomp['resid'].median()).abs().median():.5f}  |  series mean: {arpdau_ad_decomp['value'].mean():.4f}")
print(f"Coefficient of variation (resid std / series mean): {cov:.3%}")
print(f"Outliers flagged (|modified z| > {outlier_threshold}): {arpdau_ad_decomp['is_outlier'].sum()} / {len(arpdau_ad_decomp)} days")

arpdau_ad_decomp[arpdau_ad_decomp['is_outlier']][['value', 'resid', 'modified_z']]

Residual std: 0.00369  |  MAD: 0.00067  |  series mean: 0.0847
Coefficient of variation (resid std / series mean): 4.354%
Outliers flagged (|modified z| > 7): 19 / 575 days


,value,resid,modified_z
dt,,,
2025-02-05,0.082265,0.007272,7.311820
2025-02-07,0.078220,0.008363,8.408798
2025-03-10,0.090359,0.009094,9.143999
2025-03-11,0.091216,0.008735,8.782979
2025-03-12,0.089320,0.008355,8.400379
2025-04-07,0.088974,0.007400,7.440019
2025-04-08,0.089547,0.009035,9.084075
2025-05-07,0.095493,0.010965,11.025260
2025-05-08,0.091475,0.010960,11.020437


In [148]:
# Chart — observed series with flagged outlier days marked
plot_outliers(arpdau_ad_decomp, arpdau_ad_decomp['is_outlier'], title=f'{metric_col} (core) with robust-outlier days flagged').show()

### 3. Change-point detection (regime shifts)

Same PELT-based unsupervised detection as the RRW section above, on this metric's deseasonalized series.

In [149]:
# hide-output
arpdau_ad_changepoint_dates, arpdau_ad_changepoint_penalty = detect_changepoints(arpdau_ad_decomp['deseasonalized'], min_segment_days=min_segment_days)

print(f"Penalty: {arpdau_ad_changepoint_penalty:.2f}  |  Change-points detected: {len(arpdau_ad_changepoint_dates)}")
for d in arpdau_ad_changepoint_dates:
    print(' ', d.date())

Penalty: 6.35  |  Change-points detected: 7
  2025-01-15
  2025-03-06
  2025-06-19
  2025-11-21
  2026-02-04
  2026-04-30
  2026-07-09


In [150]:
# Chart — observed series with detected change-points marked
plot_changepoints(arpdau_ad_decomp, arpdau_ad_changepoint_dates, title=f'{metric_col} (core) with detected regime changes (dashed lines)').show()

### 4. Forecast (3 months ahead)

Same training-window walk-back as the RRW section, but the SARIMAX fit uses a **log-transform** (`fit_value_forecast`) instead of logit, since ARPDAU is an unbounded positive value, not a [0,1] rate. Revenue metrics are inherently noisier day-to-day than rate metrics (lumpier, driven by a smaller number of paying/high-value users), so expect a wider backtest error than the RRW/Loyalty sections.

In [151]:
# hide-output
# Walk back through detected regimes until we have >= min_training_days (set in Parameters above)
arpdau_ad_training_start = select_training_start(arpdau_ad_decomp.index, arpdau_ad_changepoint_dates, min_training_days=min_training_days)
arpdau_ad_train = arpdau_ad_series.loc[arpdau_ad_training_start:]
print(f"Training window: {arpdau_ad_training_start.date()} -> {arpdau_ad_decomp.index.max().date()} ({len(arpdau_ad_train)} days)")

Training window: 2026-04-30 -> 2026-07-29 (91 days)


In [152]:
# hide-output
arpdau_ad_backtest_mae, arpdau_ad_backtest_mape, arpdau_ad_forecast_df = fit_value_forecast(
    arpdau_ad_train, order=sarimax_order, seasonal_order=sarimax_seasonal_order,
    horizon=forecast_horizon_days, backtest_days=backtest_days, ci_alpha=ci_alpha,
)
print(f"Backtest (last {backtest_days} days of training window): MAE={arpdau_ad_backtest_mae:.5f}  MAPE={arpdau_ad_backtest_mape:.3%}")

Backtest (last 28 days of training window): MAE=0.00471  MAPE=5.430%


In [153]:
# hide-output
print(f"Forecast horizon: {arpdau_ad_forecast_df.index.min().date()} -> {arpdau_ad_forecast_df.index.max().date()}")
for h in [30, 60, 90]:
    row = arpdau_ad_forecast_df.iloc[h - 1]
    print(f"  +{h}d ({arpdau_ad_forecast_df.index[h-1].date()}): {row['forecast']:.4f}  [{row['low']:.4f}, {row['high']:.4f}]")

Forecast horizon: 2026-07-30 -> 2026-10-27
  +30d (2026-08-28): 0.0778  [0.0582, 0.1040]
  +60d (2026-09-27): 0.0852  [0.0474, 0.1534]
  +90d (2026-10-27): 0.0733  [0.0286, 0.1877]


### 5. Summary: history + forecast

In [154]:
# Chart — training window history + 90-day forecast with 80% band
plot_forecast_summary(
    arpdau_ad_series, arpdau_ad_train, arpdau_ad_forecast_df, arpdau_ad_training_start,
    title=f'{metric_col} (core) — history + {forecast_horizon_days}-day forecast', ci_alpha=ci_alpha,
).show()

In [155]:
# Summary table — forecast at key checkpoints
arpdau_ad_summary_table = arpdau_ad_forecast_df.iloc[[29, 59, 89]].reset_index().rename(columns={
    'dt': 'Date', 'forecast': 'P50 (forecast)', 'low': 'P10', 'high': 'P90',
})
arpdau_ad_summary_table.insert(0, 'Days out', [30, 60, 90])
arpdau_ad_summary_table[['P10', 'P50 (forecast)', 'P90']] = arpdau_ad_summary_table[['P10', 'P50 (forecast)', 'P90']].round(4)
arpdau_ad_summary_table

,Days out,Date,P50 (forecast),P10,P90
0,30,2026-08-28,0.0778,0.0582,0.1040
1,60,2026-09-27,0.0852,0.0474,0.1534
2,90,2026-10-27,0.0733,0.0286,0.1877


### 6. Scenario: what if core ARPDAU Ad gets a % uplift

Same standalone overlay approach as the RRW section — no upper clip here, since revenue has no natural 100% ceiling. Adjust `uplift_pct_arpdau_ad` to try different scenarios.

In [156]:
# hide-output
arpdau_ad_scenario_df = apply_uplift_scenario(arpdau_ad_forecast_df, uplift_pct_arpdau_ad, clip_upper=None)

print(f"Uplift scenario: {uplift_pct_arpdau_ad:+.1%} relative")
for h in [30, 60, 90]:
    base = arpdau_ad_scenario_df['forecast'].iloc[h - 1]
    up = arpdau_ad_scenario_df['forecast_uplift'].iloc[h - 1]
    print(f"  +{h}d: baseline={base:.4f}  uplifted={up:.4f}  (delta={up - base:+.4f})")

Uplift scenario: +2.0% relative
  +30d: baseline=0.0778  uplifted=0.0794  (delta=+0.0016)
  +60d: baseline=0.0852  uplifted=0.0869  (delta=+0.0017)
  +90d: baseline=0.0733  uplifted=0.0748  (delta=+0.0015)


In [157]:
# Chart — baseline forecast vs uplift scenario
plot_uplift_scenario(
    arpdau_ad_series, arpdau_ad_scenario_df, uplift_pct_arpdau_ad,
    title=f'{metric_col} (core) — baseline vs {uplift_pct_arpdau_ad:+.1%} uplift scenario', lookback_days=60,
).show()

### 7. Period comparison: Period 1 vs Period 2

Average core `metric_col` within each of the two shared periods defined at the top of the notebook, and the relative uplift between them.

In [158]:
# hide-output
arpdau_ad_period_comparison = compare_periods(arpdau_ad_series, period1_start, period1_end, period2_start, period2_end)

print(f"Period 1 ({period1_start} -> {period1_end}, n={arpdau_ad_period_comparison['period1_n']} days): avg {metric_col} = {arpdau_ad_period_comparison['period1_mean']:.4f}")
print(f"Period 2 ({period2_start} -> {period2_end}, n={arpdau_ad_period_comparison['period2_n']} days): avg {metric_col} = {arpdau_ad_period_comparison['period2_mean']:.4f}")
print(f"Uplift: {arpdau_ad_period_comparison['abs_diff']:+.4f}  ({arpdau_ad_period_comparison['pct_uplift']:+.1%} relative)")

Period 1 (2025-07-01 -> 2025-09-30, n=92 days): avg arpdau_ad = 0.0791
Period 2 (2026-01-01 -> 2026-06-30, n=181 days): avg arpdau_ad = 0.0914
Uplift: +0.0123  (+15.6% relative)


In [159]:
# Chart — Period 1 vs Period 2 average
plot_period_comparison(
    arpdau_ad_period_comparison, title=f'{metric_col} (core): Period 1 vs Period 2 average', y_label=metric_col,
).show()